In [6]:
import os
import time
import torch
import numpy as np
import torch.nn.functional as F
from utils import splits_classification
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid, Coauthor, CitationFull
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [7]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Net2(torch.nn.Module):
    def __init__(self, num_features, hidden, num_layers, num_classes):
        super(Net2, self).__init__()
        self.num_layers = num_layers
        self.conv = torch.nn.ModuleList()
        self.conv.append(GCNConv(num_features, hidden))
        for i in range(self.num_layers - 1):
            self.conv.append(GCNConv(hidden, hidden))
        self.lt1 = torch.nn.Linear(hidden, num_classes)

    def reset_parameters(self):
        for module in self.conv:
            module.reset_parameters()
        self.lt1.reset_parameters()

    def forward(self, x, edge_index):
        for i in range(self.num_layers):
            x = self.conv[i](x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, training=self.training)
        x = self.lt1(x)
        return F.log_softmax(x, dim = 1)

In [12]:
dataset_names = ['Physics']
for dataset_name in dataset_names:
    dataset = Coauthor(root='./dataset', name=dataset_name)
    num_classes = dataset.num_classes
    data = splits_classification(dataset[0], num_classes, exp = 'random')
    model = Net2(data.x.shape[1], 512, 2, num_classes).to(device)
    loss_fn = torch.nn.NLLLoss()
    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    avg_time = 0
    all_acc = []
    for run in range(20):
        best_val_loss = 100000
        model.reset_parameters()

        data = data.to(device)
        for epoch in range(100):
            model.train()
            optimizer.zero_grad()
            out = model(data.x, data.edge_index)
            loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                out = model(data.x, data.edge_index)
                val_loss = loss_fn(out[data.val_mask], data.y[data.val_mask])
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    #save model
                    torch.save(model.state_dict(), f'save/node_cls/baselines/baseline_{dataset_name}_random.pt')
        start = time.time()
        out = model(data.x, data.edge_index)
        avg_time += time.time() - start
        test_loss = loss_fn(out[data.test_mask], data.y[data.test_mask])
        acc = int(torch.sum(torch.argmax(out, dim=1) == data.y).item()) / len(data.y)
        all_acc.append(acc)
    top_acc= sorted(all_acc, reverse = True)[:10]

    if not os.path.exists(f"results/node_cls_baselines.csv"):
        with open(f"results/node_cls_baselines.csv", 'w') as f:
            f.write('dataset,hidden,runs,num_layers,lr,avg_time,top_10_acc,best_acc\n')

    with open(f"results/node_cls_baselines.csv", 'a') as f:
        f.write(f"{dataset_name},random,512,20,2,0.01,100,{avg_time/20},{np.mean(top_acc)} +/- {np.std(top_acc)},{top_acc[0]}\n")